# 02 - Gold tables

Splits the silver OBT into dimensional model tables (fact + dimensions).
dim_date already exists from notebook 01.

Order matters here: dimensions are built before the fact table so we have
something to reference when validating foreign keys.

*Some parts and solutions of this code was found through help from Claude*

In [0]:
import sys
import os

# Make repo root importable so we can use utils/
sys.path.append(os.path.abspath("../.."))

from pyspark.sql import functions as F
from pyspark.sql.window import Window

from utils.constants import (
    SILVER_RESULTS_OBT,
    SILVER_DIM_COUNTRY,
    GOLD_FCT_RESULTS,
    GOLD_DIM_EVENT,
    GOLD_DIM_ATHLETE,
    GOLD_DIM_COUNTRY,
)
from utils.io_helpers import read_table, write_df_to_table

silver_df = read_table(spark, SILVER_RESULTS_OBT)

## dim_event

One row per unique event. Picks the event attributes from silver.
We use distinct() to deduplicate since silver has many results per event.

In [0]:
# event_distance_value can vary slightly for the same event across years
# (e.g. course re-measured). We pick first() per event_id to guarantee uniqueness.
dim_event_df = (
    silver_df
    .groupBy("event_id")
    .agg(
        F.first("event_name").alias("event_name"),
        F.first("event_distance_value").alias("event_distance_value"),
        F.first("event_distance_unit").alias("event_distance_unit"),
        F.first("event_type").alias("event_type"),
    )
)

print(f"Rows in dim_event: {dim_event_df.count():,}")
dim_event_df.show(5, truncate=False)

write_df_to_table(dim_event_df, GOLD_DIM_EVENT)
print(f"Wrote {GOLD_DIM_EVENT}")

## dim_athlete

One row per athlete with their birth year, gender and nationality.

In [0]:
dim_athlete_df = (
    silver_df
    .groupBy("athlete_id")
    .agg(
        F.first("athlete_year_of_birth").alias("athlete_year_of_birth"),
        F.first("athlete_gender").alias("athlete_gender"),
        F.first(F.col("athlete_country")).alias("country_code"),
    )
)

print(f"Rows in dim_athlete: {dim_athlete_df.count():,}")
dim_athlete_df.show(5, truncate=False)

write_df_to_table(dim_athlete_df, GOLD_DIM_ATHLETE)
print(f"Wrote {GOLD_DIM_ATHLETE}")

## dim_country

Copy of the silver dim_country into gold. Same data, different layer -
gold should be self-contained so analysts can use it without crossing into silver.

In [0]:
dim_country_df = read_table(spark, SILVER_DIM_COUNTRY)

print(f"Rows in dim_country: {dim_country_df.count():,}")
dim_country_df.show(5, truncate=False)

write_df_to_table(dim_country_df, GOLD_DIM_COUNTRY)
print(f"Wrote {GOLD_DIM_COUNTRY}")

## Validate uniqueness of dimension primary keys

dim_event and dim_athlete must have unique primary keys for the star schema
to work. If counts don't match we have a bug in the dimension build.

In [0]:
dim_event_df = read_table(spark, GOLD_DIM_EVENT)
dim_athlete_df = read_table(spark, GOLD_DIM_ATHLETE)

event_total = dim_event_df.count()
event_unique = dim_event_df.select("event_id").distinct().count()

athlete_total = dim_athlete_df.count()
athlete_unique = dim_athlete_df.select("athlete_id").distinct().count()

print(f"dim_event:   {event_total:,} rows, {event_unique:,} unique event_id, match: {event_total == event_unique}")
print(f"dim_athlete: {athlete_total:,} rows, {athlete_unique:,} unique athlete_id, match: {athlete_total == athlete_unique}")

## fct_results

The fact table - one row per result. Joins against dim_date to get the date_ids
for start and end dates. Generates a result_id as the primary key.

In [0]:
from utils.constants import GOLD_DIM_DATE

dim_date_df = read_table(spark, GOLD_DIM_DATE)

# Start with silver and join dim_date twice - once for start, once for end
fct_df = (
    silver_df
    # Join for start_date_id - rename the joined column so we don't clash on the second join
    .join(
        dim_date_df.select(F.col("date_id").alias("start_date_id"), F.col("full_date").alias("start_match")),
        silver_df["start_date"] == F.col("start_match"),
        how="left"
    )
    .drop("start_match")
    # Join for end_date_id
    .join(
        dim_date_df.select(F.col("date_id").alias("end_date_id"), F.col("full_date").alias("end_match")),
        silver_df["end_date"] == F.col("end_match"),
        how="left"
    )
    .drop("end_match")
)

# Generate result_id as the primary key. row_number() over an unpartitioned window
# gives 1, 2, 3, ... in order. Same warning as dense_rank earlier - it's fine for our scale.
w = Window.orderBy("event_id", "athlete_id", "start_date_id")
fct_df = fct_df.withColumn("result_id", F.row_number().over(w))

# Pick the final columns in a sensible order
fct_df = fct_df.select(
    "result_id",
    "event_id",
    "athlete_id",
    F.col("athlete_country").alias("country_code"),
    "start_date_id",
    "end_date_id",
    "event_type",
    "performance_seconds",
    "performance_distance",
    F.col("performance_unit"),
    "age_at_event",
    F.col("event_num_finishers"),
)

print(f"Rows in fct_results: {fct_df.count():,}")
fct_df.show(5, vertical=True, truncate=False)

write_df_to_table(fct_df, GOLD_FCT_RESULTS)
print(f"Wrote {GOLD_FCT_RESULTS}")